In [1]:
# TensorFlow/Keras EDA - Deep Learning for NLP
# This notebook covers text preprocessing and vectorization using TensorFlow

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.19.0


### 1. SAMPLE DATA PREPARATION

In [2]:
sample_texts = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with multiple layers.",
    "Natural language processing helps computers understand human language.",
    "Machine learning models require large amounts of training data.",
    "Neural networks are inspired by biological neural systems.",
    "Text preprocessing is crucial for NLP tasks.",
    "Deep learning has revolutionized computer vision and NLP.",
    "Artificial intelligence is transforming industries worldwide."
]

labels = [1, 1, 1, 1, 1, 0, 1, 1]  # Example binary labels

print("=" * 70)
print("1. SAMPLE DATA")
print("=" * 70)
print(f"\nTotal documents: {len(sample_texts)}")
for i, text in enumerate(sample_texts[:3], 1):
    print(f"  {i}. {text}")

1. SAMPLE DATA

Total documents: 8
  1. Machine learning is a subset of artificial intelligence.
  2. Deep learning uses neural networks with multiple layers.
  3. Natural language processing helps computers understand human language.


### 2. TOKENIZATION WITH KERAS

In [3]:
print("\n" + "=" * 70)
print("2. TOKENIZATION WITH KERAS")
print("=" * 70)

tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(sample_texts)

# Get vocabulary statistics
vocab_size = len(tokenizer.word_index) + 1
print(f"\nVocabulary size: {vocab_size}")
print("-" * 30)
print(f"\nWord to index mapping (first 20 words):")
word_index = dict(sorted(tokenizer.word_index.items(), key=lambda x: x[1])[:20])
print("\n{:<20} {:<10}".format("Word", "Index"))
print("-" * 30)
for word, idx in word_index.items():
    print("{:<20} {:<10}".format(word, idx))


2. TOKENIZATION WITH KERAS

Vocabulary size: 49
------------------------------

Word to index mapping (first 20 words):

Word                 Index     
------------------------------
<OOV>                1         
learning             2         
is                   3         
neural               4         
machine              5         
of                   6         
artificial           7         
intelligence         8         
deep                 9         
networks             10        
language             11        
nlp                  12        
a                    13        
subset               14        
uses                 15        
with                 16        
multiple             17        
layers               18        
natural              19        
processing           20        


### 3. SEQUENCE CONVERSION

In [4]:
print("\n" + "=" * 70)
print("3. SEQUENCE CONVERSION (TEXT TO SEQUENCES)")
print("=" * 70)

sequences = tokenizer.texts_to_sequences(sample_texts)
print(f"\nOriginal text 1: '{sample_texts[0]}'")
print(f"Sequence 1: {sequences[0]}")

# Decode sequence back to text
reverse_word_index = dict([(value, key) for (key, value) in tokenizer.word_index.items()])

def decode_sequence(seq):
    return ' '.join([reverse_word_index.get(i, '?') for i in seq])

print(f"Decoded sequence 1: '{decode_sequence(sequences[0])}'")


3. SEQUENCE CONVERSION (TEXT TO SEQUENCES)

Original text 1: 'Machine learning is a subset of artificial intelligence.'
Sequence 1: [5, 2, 3, 13, 14, 6, 7, 8]
Decoded sequence 1: 'machine learning is a subset of artificial intelligence'


### 4. SEQUENCE PADDING

In [5]:
print("\n" + "=" * 70)
print("4. SEQUENCE PADDING (STANDARDIZING SEQUENCE LENGTH)")
print("=" * 70)

# Find max length
max_length = max(len(seq) for seq in sequences)
print(f"\nMax sequence length: {max_length}")

# Pad sequences
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')
print(f"Padded sequence shape: {padded_sequences.shape}")
print(f"\nOriginal sequence 0: {sequences[0]}")
print(f"Padded sequence 0:   {padded_sequences[0]}")
print(f"\nOriginal sequence 1: {sequences[1]}")
print(f"Padded sequence 1:   {padded_sequences[1]}")


4. SEQUENCE PADDING (STANDARDIZING SEQUENCE LENGTH)

Max sequence length: 9
Padded sequence shape: (8, 9)

Original sequence 0: [5, 2, 3, 13, 14, 6, 7, 8]
Padded sequence 0:   [ 5  2  3 13 14  6  7  8  0]

Original sequence 1: [9, 2, 15, 4, 10, 16, 17, 18]
Padded sequence 1:   [ 9  2 15  4 10 16 17 18  0]


### 5. WORD FREQUENCY ANALYSIS

In [6]:
print("\n" + "=" * 70)
print("5. WORD FREQUENCY ANALYSIS")
print("=" * 70)

word_counts = tokenizer.word_counts
top_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:15]

print(f"\nTotal unique words: {len(word_counts)}")
print(f"\nTop 15 most frequent words:")
print("{:<20} {:<10}".format("Word", "Frequency"))
print("-" * 30)
for word, count in top_words:
    print("{:<20} {:<10}".format(word, count))


5. WORD FREQUENCY ANALYSIS

Total unique words: 47

Top 15 most frequent words:
Word                 Frequency 
------------------------------
learning             4         
is                   3         
neural               3         
machine              2         
of                   2         
artificial           2         
intelligence         2         
deep                 2         
networks             2         
language             2         
nlp                  2         
a                    1         
subset               1         
uses                 1         
with                 1         


### 6. DOCUMENT LENGTH ANALYSIS

In [7]:
print("\n" + "=" * 70)
print("6. DOCUMENT LENGTH ANALYSIS")
print("=" * 70)

doc_lengths = [len(seq) for seq in sequences]
print(f"\nDocument length statistics:")
print(f"  Min length: {min(doc_lengths)}")
print(f"  Max length: {max(doc_lengths)}")
print(f"  Mean length: {np.mean(doc_lengths):.2f}")
print(f"  Median length: {np.median(doc_lengths):.2f}")
print(f"  Std Dev: {np.std(doc_lengths):.2f}")



6. DOCUMENT LENGTH ANALYSIS

Document length statistics:
  Min length: 6
  Max length: 9
  Mean length: 7.75
  Median length: 8.00
  Std Dev: 0.83


### 7. VECTORIZATION WITH DENSE LAYERS

In [8]:
print("\n" + "=" * 70)
print("7. MANUAL VECTORIZATION (TF-IDF STYLE)")
print("=" * 70)

# Create TF-IDF style vectors
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec = TfidfVectorizer(max_features=100)
tfidf_matrix = tfidf_vec.fit_transform(sample_texts).toarray()

print(f"tfidf_matrix: {tfidf_matrix[0]}")
print(f"\nTF-IDF matrix shape: {tfidf_matrix.shape}")
# print(f"Feature names (first 15): {tfidf_vec.get_feature_names_out()[:15]}")



7. MANUAL VECTORIZATION (TF-IDF STYLE)
tfidf_matrix: [0.         0.         0.         0.38516288 0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.38516288
 0.33236397 0.         0.         0.         0.29140994 0.38516288
 0.         0.         0.         0.         0.         0.
 0.38516288 0.         0.         0.         0.         0.45957878
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.        ]

TF-IDF matrix shape: (8, 46)


### 8. WORD EMBEDDING PREPARATION

In [9]:
print("\n" + "=" * 70)
print("8. PREPARING FOR WORD EMBEDDINGS")
print("=" * 70)

print(f"\nWord embedding input shape: {padded_sequences.shape}")
print(f"Each sample will be: (sequence_length={max_length},)")
print(f"After embedding layer (embed_dim=32): ({max_length}, 32)")

# Example: Create a simple embedding model
embedding_model = keras.Sequential([
    keras.layers.Embedding(input_dim=vocab_size, output_dim=32,
                          input_length=max_length, name='embedding'),
    keras.layers.Flatten(name='flatten'),
    keras.layers.Dense(64, activation='relu', name='dense1'),
    keras.layers.Dense(32, activation='relu', name='dense2'),
    keras.layers.Dense(1, activation='sigmoid', name='output')
])

print(f"\nEmbedding model architecture:")
embedding_model.summary()


8. PREPARING FOR WORD EMBEDDINGS

Word embedding input shape: (8, 9)
Each sample will be: (sequence_length=9,)
After embedding layer (embed_dim=32): (9, 32)

Embedding model architecture:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense2 (Dense)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### 9. ONE-HOT ENCODING ALTERNATIVE

In [10]:
print("\n" + "=" * 70)
print("9. ONE-HOT ENCODING ALTERNATIVE")
print("=" * 70)

# Create one-hot encoded matrix (simpler alternative)
onehot_matrix = np.zeros((len(sample_texts), vocab_size))
for i, seq in enumerate(sequences):
    for idx in seq:
        if idx < vocab_size:
            onehot_matrix[i, idx] = 1

print(f"\nOne-hot encoded matrix shape: {onehot_matrix.shape}")
print(f"First document one-hot (first 10 dims): {onehot_matrix[0][:10]}")
print(f"Non-zero elements in first doc: {np.count_nonzero(onehot_matrix[0])}")



9. ONE-HOT ENCODING ALTERNATIVE

One-hot encoded matrix shape: (8, 49)
First document one-hot (first 10 dims): [0. 0. 1. 1. 0. 1. 1. 1. 1. 0.]
Non-zero elements in first doc: 8
